# 02 · Entrenar la política ACT de agarre (LeRobot) y comprobarla sin simulador

**Qué aprende.** Dado el RGB de la cámara cenital, la **segmentación que marca qué pieza coger** y las articulaciones, ACT predice bloques de
50 consignas de articulación (aproximar → bajar → cerrar → subir). **No** aprende a llevar la pieza a la caja: eso es programado
(`docs/02_arquitectura.md`: arquitectura híbrida y por qué).

**Entrada.** `data/bin_pick_v1/` grabado con `tools/collect_demos.py` (≈ 500 episodios), comprimido en zip y subido a Drive.
**Salida.** `pretrained_model` + `binpick_info.json`, que se usa con `python 18_voice_bin_picking.py --policy act:<carpeta>`.

**Tiempo orientativo (sin medir).** 500 episodios ≈ 85 000 fotogramas; 50 000 pasos con lote 16 en una T4 son varias horas. Empieza con una
pasada corta (`STEPS = 2000`) para ver que la pérdida baja antes de lanzar la larga.

**Variantes que conviene comparar** (el mismo dataset sirve para todas): con `OBS_MODE`, `rgb_mask` (recomendada),
`rgb_only` (control: no sabe qué pieza; debería fallar con varias piezas) y `mask_only` (¿basta la segmentación sola?). y además `NO_VAE` y `NO_STATE` (sin las articulaciones como entrada: en el demo 06 la política con cámara + estado copiaba la pose de su propio brazo). No se ha medido cuál gana:
ése es el experimento.

**Importante.** Un buen error offline **no** garantiza que agarre. Lo que decide es `tools/eval_sorting.py` en Isaac.

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "· CUDA:", torch.cuda.is_available())

In [ ]:
!pip -q install "lerobot==0.4.4"      # la versión con la que se grabó el dataset (formato v3)
import lerobot, torch; print("lerobot", getattr(lerobot, "__version__", "?"), "· torch", torch.__version__)

## Datos y parámetros

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ZIP        = "/content/drive/MyDrive/piezas/bin_pick_v1.zip"    # <- ajusta
OBS_MODE   = "rgb_mask"        # rgb_mask | rgb_only | mask_only
STEPS      = 50000             # empieza con 2000 para una prueba
BATCH      = 16                # baja a 8 si se queda sin memoria
CHUNK, ACTION_STEPS = 50, 25
NO_VAE     = False             # segundo experimento: True
NO_STATE   = False             # tercer experimento: True = sin articulaciones como entrada (anti copia de la pose del propio brazo)
TRAIN_EPISODES = "0-449"       # 450-499 quedan sin ver para la comprobación offline
OUT        = f"/content/outputs/act_{OBS_MODE}"
import pathlib
!rm -rf /content/data && mkdir -p /content/data && unzip -q "{ZIP}" -d /content/data
DATA = next(p.parent.parent for p in pathlib.Path("/content/data").rglob("info.json") if p.parent.name == "meta")
print("dataset:", DATA)

In [ ]:
# Comprobación rápida del dataset (sólo lectura: claves, formas, NaN, unidades, que las acciones se mueven)
import json, glob, numpy as np, pandas as pd, pathlib
info = json.load(open(DATA / "meta" / "info.json"))
print("episodios:", info["total_episodes"], "· fotogramas:", info["total_frames"], "· fps:", info["fps"])
for k in ("observation.images.overhead", "observation.images.target", "observation.state", "action"):
    assert k in info["features"], f"falta {k}"
bi = json.load(open(DATA / "meta" / "binpick_info.json")); print("recorte:", bi["roi"], "→", bi["out_hw"], "· segmentador:", bi["segmenter"], "· ruido:", bi["noise"])
df = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(DATA / "data/*/*.parquet")))], ignore_index=True)
S = np.stack(df["observation.state"]); A = np.stack(df["action"])
assert np.isfinite(S).all() and np.isfinite(A).all(), "NaN en estado o acción"
assert (np.abs(A[:, :5]) <= 101).all() and (A[:, 5] >= -1).all() and (A[:, 5] <= 101).all(), "acciones fuera de unidades LeRobot"
print("desviación típica de la acción por articulación:", A.std(0).round(1), "· |acción−estado| medio:", np.abs(A - S).mean().round(2))
assert np.abs(A - S).mean() > 0.02, "la acción es una copia del estado"
ep = df["episode_index"].to_numpy(); L = np.bincount(ep)
print("longitud de episodio (fotogramas): min", L.min(), "mediana", int(np.median(L)), "max", L.max())
log = DATA / "meta" / "binpick_episodes.jsonl"
if log.exists():
    rows = [json.loads(l) for l in open(log)]; ok = sum(r.get("success", False) for r in rows)
    print(f"experto: {ok}/{len(rows)} intentos válidos ({100*ok/len(rows):.0f} %)")

In [ ]:
%%writefile train_act.py
"""Entrena ACT (LeRobot) sobre un dataset de demostraciones de este demo. Sirve en Colab (Linux) y en local. NO lo lanza nadie por ti.

    python train_act.py --dataset-root data/bin_pick_v1 --steps 50000 --batch 16 --output outputs/act_rgbmask_v1
    python train_act.py ... --obs-mode rgb_only                       # variante de control (sin saber qué pieza)
    python train_act.py ... --no-state                                # sin articulaciones como entrada (anti copia de pose)
    python train_act.py ... --episodes 0-449                          # deja 450-499 para offline_eval_policy.py
    python train_act.py ... --init-from outputs/act_v1/checkpoints/last/pretrained_model --steps 20000   # afinar con más datos

Variantes de entrada (``--obs-mode``; ver ``docs/02_arquitectura.md``): ``rgb_mask`` (recomendada: RGB + segmentación del objetivo),
``rgb_only`` (control) y ``mask_only``. Son subconjuntos de lo grabado: no hay que volver a grabar para compararlas.

Qué decide este script y por qué (cada valor es una elección, no un resultado medido en este dataset):

* ``chunk_size 50`` a 30 Hz = 1,7 s por bloque, lo que dura aproximar, bajar y cerrar; ``n_action_steps 25`` re-observa a mitad de bloque
  (lazo más cerrado que el de 100 de LeRobot por defecto). La política que funcionó en el demo 06 fue de lazo abierto y se hundía con
  cualquier desvío;
* VAE activado como en LeRobot (con 500 episodios ya no sobra: el demo 06 lo apagó con sólo 48). Prueba ``--no-vae`` como segundo experimento;
* tasa de aprendizaje 1e-5 (la de ACT, afinada para ResNet preentrenado), sin aumentos de imagen (``ImageTransforms`` también tocaría
  los colores de la máscara de segmentación).

Escribe ``<output>/`` con los checkpoints de LeRobot y, junto a cada ``pretrained_model``, ``binpick_info.json`` (modo de observación,
recorte, segmentador, unidades) que ``binpick/policy.py`` comprueba al cargar. Necesita ``lerobot==0.4.4`` (con el que se grabó).
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

IMAGE_KEYS = {"overhead": "observation.images.overhead", "target": "observation.images.target"}
STATE_KEY, ACTION_KEY = "observation.state", "action"
OBS_MODES = {"rgb_mask": ("overhead", "target"), "rgb_only": ("overhead",), "mask_only": ("target",)}


def parse_episodes(text: str | None) -> list[int] | None:
    if not text:
        return None
    out: list[int] = []
    for part in text.split(","):
        if "-" in part:
            a, b = part.split("-")
            out.extend(range(int(a), int(b) + 1))
        else:
            out.append(int(part))
    return out


def patch_last_checkpoint_windows() -> None:
    """LeRobot enlaza ``checkpoints/last`` con un symlink; en Windows sin permisos falla. Se sustituye por una copia."""
    import shutil

    import lerobot.scripts.lerobot_train as lerobot_train
    import lerobot.utils.train_utils as train_utils
    from lerobot.utils.constants import LAST_CHECKPOINT_LINK

    def update_last_checkpoint(checkpoint_dir):
        checkpoint_dir = Path(checkpoint_dir)
        link = checkpoint_dir.parent / LAST_CHECKPOINT_LINK
        if link.is_symlink() or link.is_file():
            link.unlink()
        elif link.is_dir():
            shutil.rmtree(link)
        shutil.copytree(checkpoint_dir, link)
        return link

    train_utils.update_last_checkpoint = update_last_checkpoint
    lerobot_train.update_last_checkpoint = update_last_checkpoint


def build_cfg(args):
    """``TrainPipelineConfig`` de LeRobot para estos argumentos (sin entrenar nada)."""
    from lerobot.configs.default import DatasetConfig
    from lerobot.configs.train import TrainPipelineConfig
    from lerobot.configs.types import FeatureType
    from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
    from lerobot.datasets.utils import dataset_to_policy_features
    from lerobot.policies.act.configuration_act import ACTConfig

    root = Path(args.dataset_root).resolve()
    repo_id = args.repo_id or root.name
    meta = LeRobotDatasetMetadata(repo_id, root=str(root))
    pf = dataset_to_policy_features(meta.features)
    # el estado articular es una entrada más; sin él (``--no-state``) la red no puede copiar la pose de su propio brazo
    # (la «confusión causal» que hundió a las políticas con cámara + estado del demo 06). LeRobot 0.4.4 lee igualmente la
    # clave del lote, que sigue estando en el dataset: sólo se saca de las entradas de la red.
    keep = {*(IMAGE_KEYS[s] for s in OBS_MODES[args.obs_mode])} | (set() if args.no_state else {STATE_KEY})
    missing = keep - set(pf)
    if missing:
        raise SystemExit(f"el dataset no tiene {sorted(missing)}; ¿se grabó con tools/collect_demos.py?")
    output_features = {k: v for k, v in pf.items() if v.type is FeatureType.ACTION}
    input_features = {k: v for k, v in pf.items() if k in keep}
    policy = ACTConfig(
        input_features=input_features, output_features=output_features, device=args.device,
        chunk_size=args.chunk, n_action_steps=min(args.action_steps, args.chunk), use_vae=not args.no_vae,
        kl_weight=args.kl_weight, optimizer_lr=args.lr, push_to_hub=False,
        pretrained_backbone_weights=None if args.no_pretrained_backbone else "ResNet18_Weights.IMAGENET1K_V1")
    if args.init_from:
        policy.pretrained_path = Path(args.init_from)
    out = Path(args.output).resolve()
    return TrainPipelineConfig(
        dataset=DatasetConfig(repo_id=repo_id, root=str(root), episodes=parse_episodes(args.episodes)),
        policy=policy, output_dir=out, job_name=out.name, seed=args.seed, batch_size=args.batch, steps=args.steps,
        num_workers=args.workers, save_freq=args.save_freq or max(1000, args.steps // 5), log_freq=args.log_freq,
        eval_freq=args.steps + 1)       # LeRobot no tiene aquí un entorno de evaluación: se evalúa con tools/eval_sorting.py


def write_info(args, root: Path, out: Path) -> None:
    src = root / "meta" / "binpick_info.json"
    info = json.loads(src.read_text(encoding="utf-8")) if src.is_file() else {}
    info.update({"obs_mode": args.obs_mode, "trained_on": root.name, "steps": args.steps, "batch": args.batch, "chunk": args.chunk,
                 "n_action_steps": min(args.action_steps, args.chunk), "use_vae": not args.no_vae, "use_state": not args.no_state, "lr": args.lr,
                 "episodes": args.episodes or "todos", "lerobot": _lerobot_version()})
    text = json.dumps(info, indent=2, ensure_ascii=False)
    (out / "binpick_info.json").write_text(text, encoding="utf-8")
    for ck in (out / "checkpoints").glob("*/pretrained_model"):
        (ck / "binpick_info.json").write_text(text, encoding="utf-8")


def _lerobot_version() -> str:
    try:
        import lerobot
        return getattr(lerobot, "__version__", "?")
    except Exception:  # noqa: BLE001
        return "?"


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset-root", required=True)
    ap.add_argument("--repo-id", default=None)
    ap.add_argument("--obs-mode", default="rgb_mask", choices=sorted(OBS_MODES))
    ap.add_argument("--steps", type=int, default=50000)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--chunk", type=int, default=50)
    ap.add_argument("--action-steps", type=int, default=25)
    ap.add_argument("--no-vae", action="store_true")
    ap.add_argument("--no-state", action="store_true", help="no dar las articulaciones como entrada (experimento anti copia de pose)")
    ap.add_argument("--kl-weight", type=float, default=10.0)
    ap.add_argument("--episodes", default=None, help="p. ej. 0-449 (deja el resto sin ver para la evaluación offline)")
    ap.add_argument("--output", default=None, help="carpeta nueva (LeRobot no reutiliza una existente)")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--save-freq", type=int, default=None)
    ap.add_argument("--log-freq", type=int, default=200)
    ap.add_argument("--init-from", default=None, help="carpeta pretrained_model desde la que continuar (afinado)")
    ap.add_argument("--no-pretrained-backbone", action="store_true", help="no descargar los pesos ImageNet del ResNet (sin red)")
    args = ap.parse_args()

    import torch

    if args.device == "cuda" and not torch.cuda.is_available():
        print("CUDA no está disponible: se entrena en CPU (muy lento).")
        args.device = "cpu"
    args.output = args.output or f"outputs/act_{args.obs_mode}_{time.strftime('%Y%m%d_%H%M')}"
    if Path(args.output).exists():
        print(f"ERROR: {args.output} ya existe (LeRobot no reutiliza carpetas); usa otro --output")
        return 1
    cfg = build_cfg(args)
    print(f"dataset {cfg.dataset.root} · entradas: {sorted(cfg.policy.input_features)} · bloque {args.chunk}/{args.action_steps} · "
          f"VAE {'no' if args.no_vae else 'sí'} · {args.steps} pasos × lote {args.batch} · {args.device}", flush=True)
    if sys.platform == "win32":
        patch_last_checkpoint_windows()
    from lerobot.scripts.lerobot_train import train

    t0 = time.time()
    train(cfg)
    write_info(args, Path(args.dataset_root).resolve(), Path(args.output).resolve())
    print(f"entrenado en {(time.time() - t0) / 60:.1f} min · modelo: {Path(args.output) / 'checkpoints' / 'last' / 'pretrained_model'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
flags = ("--no-vae " if NO_VAE else "") + ("--no-state" if NO_STATE else "")
!python train_act.py --dataset-root "{DATA}" --obs-mode {OBS_MODE} --steps {STEPS} --batch {BATCH} --chunk {CHUNK} --action-steps {ACTION_STEPS} --episodes {TRAIN_EPISODES} --output "{OUT}" {flags}

## Comprobación sin simulador (episodios que la red no vio)

In [ ]:
%%writefile offline_eval_policy.py
"""Comprobación SIN simulador de una política ACT entrenada: ¿predice las acciones del experto en episodios que no vio?

    python offline_eval_policy.py --policy outputs/act_v1/checkpoints/last/pretrained_model --dataset-root data/bin_pick_v1 --episodes 450-499

Para cada fotograma de los episodios indicados (que NO deben haber entrado en el entrenamiento: ``train_act.py --episodes 0-449``)
predice un bloque de acciones y lo compara con las acciones grabadas del experto, en unidades de LeRobot (−100…100; la pinza 0…100).
Se da el error absoluto medio por articulación y por paso del bloque, y se compara con dos líneas base triviales:

* «copiar el estado» (la acción = la posición actual: lo que haría una política que se queda quieta);
* «mantener la última acción» (la acción de hace un paso).

**Qué dice y qué no.** Que el error sea mucho menor que el de «copiar el estado» a horizontes largos es una buena señal de que la red
aprendió algo. NO dice que agarre: un error pequeño en la trayectoria no garantiza un agarre por fricción de 25 mm, y una política con
error pequeño puede fallar en lazo cerrado por desviaciones acumuladas (demo 06). El juez es ``tools/eval_sorting.py`` en Isaac.

No importa ``binpick``: sólo ``lerobot`` y numpy, para poder usarlo tal cual en Colab.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

STATE_KEY, ACTION_KEY = "observation.state", "action"
HORIZONS = (0, 5, 10, 25, 49)


def parse_episodes(text: str) -> list[int]:
    out: list[int] = []
    for part in text.split(","):
        if "-" in part:
            a, b = part.split("-")
            out.extend(range(int(a), int(b) + 1))
        else:
            out.append(int(part))
    return out


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--policy", required=True, help="carpeta pretrained_model")
    ap.add_argument("--dataset-root", required=True)
    ap.add_argument("--repo-id", default=None)
    ap.add_argument("--episodes", required=True, help="p. ej. 450-499")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--stride", type=int, default=5, help="un fotograma de cada N")
    ap.add_argument("--out", default=None, help="fichero .json de salida")
    args = ap.parse_args()

    import torch
    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    from lerobot.policies.act.modeling_act import ACTPolicy
    from lerobot.policies.factory import make_pre_post_processors

    device = args.device if (args.device != "cuda" or torch.cuda.is_available()) else "cpu"
    root = Path(args.dataset_root).resolve()
    repo_id = args.repo_id or root.name
    cfg = PreTrainedConfig.from_pretrained(args.policy)
    cfg.device = device
    policy = ACTPolicy.from_pretrained(args.policy, config=cfg).to(device).eval()
    pre, post = make_pre_post_processors(policy_cfg=policy.config, pretrained_path=args.policy,
                                         preprocessor_overrides={"device_processor": {"device": device}})
    chunk = policy.config.chunk_size
    fps = 30
    ds = LeRobotDataset(repo_id, root=str(root), episodes=parse_episodes(args.episodes),
                        delta_timestamps={ACTION_KEY: [i / fps for i in range(chunk)]})
    # error por [articulación, paso del bloque] acumulado, y de las líneas base
    err = np.zeros((6, chunk))
    base_state = np.zeros((6, chunk))
    base_last = np.zeros((6, chunk))
    n = 0
    for i in range(0, len(ds), args.stride):
        item = ds[i]
        valid = ~item["action_is_pad"].numpy() if "action_is_pad" in item else np.ones(chunk, bool)
        batch = {k: (v.unsqueeze(0) if hasattr(v, "unsqueeze") else v) for k, v in item.items()
                 if k.startswith("observation.")}
        with torch.no_grad():
            pred = post(policy.predict_action_chunk(pre(batch)))            # (1, chunk, 6) desnormalizado
        pred = pred.squeeze(0).cpu().numpy()
        truth = item[ACTION_KEY].numpy()
        state = item[STATE_KEY].numpy()
        for t in range(chunk):
            if not valid[t]:
                continue
            err[:, t] += np.abs(pred[t] - truth[t])
            base_state[:, t] += np.abs(state - truth[t])
            base_last[:, t] += np.abs(truth[0] - truth[t])
        n += 1
    names = ["pan", "lift", "elbow", "wrist_flex", "wrist_roll", "pinza"]
    rep = {"episodios": args.episodes, "fotogramas_evaluados": n, "bloque": chunk, "unidades": "LeRobot (−100…100; pinza 0…100)",
           "error_medio_por_articulación_y_paso": {}}
    for h in [x for x in HORIZONS if x < chunk]:
        rep["error_medio_por_articulación_y_paso"][f"paso {h}"] = {
            "política": dict(zip(names, (err[:, h] / n).round(2).tolist())),
            "copiar_el_estado": dict(zip(names, (base_state[:, h] / n).round(2).tolist())),
            "mantener_la_última_acción": dict(zip(names, (base_last[:, h] / n).round(2).tolist())),
        }
    pol, cp = err.mean() / n, base_state.mean() / n
    rep["resumen"] = {"error_política": round(float(pol), 3), "error_copiar_estado": round(float(cp), 3),
                      "mejora_frente_a_copiar_el_estado": round(float(1 - pol / cp), 3) if cp else None}
    print(json.dumps(rep, indent=1, ensure_ascii=False))
    if args.out:
        Path(args.out).write_text(json.dumps(rep, indent=1, ensure_ascii=False), encoding="utf-8")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
!python offline_eval_policy.py --policy "{OUT}/checkpoints/last/pretrained_model" --dataset-root "{DATA}" --episodes 450-499 --stride 5 --out "{OUT}/offline_eval.json"

Cómo leerlo: el error de la política a los pasos 25 y 49 del bloque tiene que ser **claramente menor** que el de «copiar el estado». Si no, la red no ha
aprendido a anticipar: más pasos, más episodios o revisar el dataset (`tools/validate_dataset.py`). Si es menor, el siguiente juez es Isaac.

In [ ]:
import shutil, json
shutil.make_archive("/content/pretrained_model", "zip", f"{OUT}/checkpoints/last/pretrained_model")
shutil.copy("/content/pretrained_model.zip", f"/content/drive/MyDrive/piezas/act_{OBS_MODE}.zip")
print(json.load(open(f"{OUT}/checkpoints/last/pretrained_model/binpick_info.json")))
from google.colab import files; files.download("/content/pretrained_model.zip")

## Usarlo en el demo

```powershell
# descomprime en demos/09_voice_bin_picking/outputs/act_rgb_mask/pretrained_model
python tools/eval_sorting.py --headless --seeds 20 --policy act:outputs/act_rgb_mask/pretrained_model     # el juez real
python 18_voice_bin_picking.py --policy act+guion:outputs/act_rgb_mask/pretrained_model --voice --open     # demo con respaldo del guion (se etiqueta)
```

## Más datos / afinado

Si la política falla en situaciones concretas (pilas, piezas cerca de la pared), graba **más** episodios de ese tipo
(`collect_demos.py --resume --layout apiladas --first-seed 5000`), sube el dataset ampliado y continúa desde el checkpoint:

```
!python train_act.py --dataset-root "{DATA}" --obs-mode {OBS_MODE} --steps 20000 --init-from "{OUT}/checkpoints/last/pretrained_model" --output /content/outputs/act_v2
```